<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/03_PCMCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the pcmci family of causal discovery algorithms on the correctedv3 dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATASET_PATH = '/content/drive/MyDrive/ml/CORRECTEDv3/all_cities_combined_v3.parquet'

In [4]:
!pip install tigramite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.7/314.7 kB 19.2 MB/s eta 0:00:00


# Objective

- PCMCI+ — single pooled temporal causal discovery ,  Output : Directed lagged graph
- J-PCMCI+ — joint multi-city temporal causal discovery , Output : Joint graph

# why reduced set of features

- PCMCI+ runtime scales roughly as O(N² × T) for the skeleton phase and O(N³ × T) for the MCI test phase, where N is the number of variables and T is the time-series length.

- Going from N=28 to N=10 reduces the MCI phase by a factor of (28/10)³ ≈ 22×.

-  ParCorr conditions on linear combinations of the conditioning set — if we include both X and a monotone transform of X (like workload_causal and workload_capped), the conditioning sets become linearly dependent and the partial correlation tests lose meaning.

In [5]:
import pandas as pd
import numpy as np
import tigramite
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

# 1. Load the dataset
dataset_path = '/content/drive/MyDrive/ml/CORRECTEDv3/all_cities_combined_v3.parquet'
df = pd.read_parquet(dataset_path)

parcorr - The theoretical backing is the faithfulness condition (Spirtes et al., 2000): PCMCI+ assumes the graph is faithful to the distribution, but including functional redundancies violates this because the conditional independences you observe are artefacts of the transforms, not of the causal structure.

In [ ]:
# 2. Define the features requested
# requested_features = [
#     'workload_causal', 'workload_capped', 'high_load', 'overloaded',
#     'pickup_destination_distance', 'batch_size', 'batch_rank_dispatch',
#     'batch_rank_capped', 'late_batch', 'extreme_batch', 'hour_sin', 'hour_cos',
#     'day_sin', 'day_cos', 'is_weekend', 'is_holiday', 'is_holiday_eve',
#     'spatial_congestion_index_daily', 'spatial_congestion_index_rolling7',
#     'spatial_congestion_norm', 'courier_local_load', 'WSI', 'precipitation',
#     'temperature_2m', 'windspeed_10m', 'is_trajectory_available', 'typecode_cb', 'eta_mins'
# ]  original set

requested_features = [
    'workload_causal', 'workload_capped', 'high_load', 'overloaded',]

In [ ]:




# Handle wildcard for typecode_grouped_*
grouped_features = [col for col in df.columns if col.startswith('typecode_grouped_')]
requested_features.extend(grouped_features)

# Check which features actually exist in the dataframe to avoid KeyErrors
features = [f for f in requested_features if f in df.columns]
missing = set(requested_features) - set(features)
if missing:
    print(f"Skipping missing columns: {missing}")

# Filter dataframe and handle missing values for PCMCI
selected_df = df[features].dropna().reset_index(drop=True)

# 3. Initialize Tigramite dataframe object
var_names = selected_df.columns.tolist()
data_values = selected_df.values
link_matrix_data = pp.DataFrame(data_values, var_names=var_names)

# 4. Initialize and Run PCMCI+
# Fixed: Changed significance='ait' to 'analytic'
parcorr = ParCorr(significance='analytic')
pcmci = PCMCI(dataframe=link_matrix_data, cond_ind_test=parcorr, verbosity=1)

# Run pcmci_plus
results = pcmci.run_pcmciplus(tau_max=2, pc_alpha=0.05)

# Display summary results
pcmci.print_significant_links(
    p_matrix=results['p_matrix'],
    val_matrix=results['val_matrix'],
    alpha_level=0.05
)

Skipping missing columns: {'spatial_congestion_index_rolling7', 'spatial_congestion_index_daily'}

##
## Step 1: PC1 algorithm for selecting lagged conditions
##

Parameters:
independence test = par_corr
tau_min = 1
tau_max = 2
pc_alpha = [0.05]
max_conds_dim = None
max_combinations = 1


